# Postnatal Multiome analysis 

# Load packages

In [ ]:
library(Signac)
library(Seurat)
library(rtracklayer)
library(dplyr)
library(Matrix)
library(ggplot2)
library(cowplot)
require(biomaRt)
library(DoubletFinder)
library(RColorBrewer)
library(stringr)
set.seed(1234)

In [ ]:
my_c <- c('#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4', '#f5674a', '#C79FEF', '#DAA520', '#A0522D')

# Data loading

 This genes are the ones used to perform the aligment on CellRanger using the costumized genome reference. It contains teh CreERT2 gene and e GFP gene.

In [ ]:
annotation <- import('genes.gtf')

In [ ]:
annotation$gene_biotype[1:1780455] = annotation$gene_type[1:1780455]
seqlevelsStyle(annotation) <- "UCSC" #Support for other than Ensembl seqlevel style

In [ ]:
#load fragments data
P11_sample1_frags <- 'P11_sample1_atac_fragments.tsv.gz'

P11_sample2_frags <- 'P11_sample2_atac_fragments.tsv.gz'

P11_sample3_frags <- 'P11_sample3_atac_fragments.tsv.gz'

P60_sample1_frags <- 'P60_sample1_atac_fragments.tsv.gz'

P60_sample2_frags <- 'P60_sample2_atac_fragments.tsv.gz'

P60_TTR_sample3_frags <- 'P60_TTR_sample3_atac_fragments.tsv.gz'

In [ ]:
P11_sample1  <- readRDS('P11_sample1_multiome.rds')

P11_sample2  <- readRDS('P11_sample2_multiome.rds')

P11_sample3  <- readRDS('P11_sample3_multiome.rds')

P60_sample1  <- readRDS('P60_sample1_multiome.rds')

P60_sample2  <- readRDS('P60_sample2_multiome.rds')

P60_TTR_sample3  <- readRDS('P60_TTR_sample3_multiome.rds')

In [ ]:
multiome_objs <- c(P11_sample1, P11_sample2, P11_sample3, P60_sample1, P60_sample2, P60_TTR_sample3)

In [ ]:
sample_names <- c('P11_sample1', 'P11_sample2', 'P11_sample3', 'P60_sample1', 'P60_sample2', 'P60_TTR_sample3')

In [ ]:
for (i in 1:length(multiome_objs)){
    batch <- tail(str_split(sample_names[[i]], pattern='_')[[1]], n=1)
    print(batch)
    multiome_objs[[i]]$Batch <- batch
}


In [ ]:
merged <- merge(multiome_objs[[1]], y = multiome_objs[2:6], add.cell.ids = sample_names, project = 'Multiome_PostNaltal')

In [ ]:
merged <- readRDS('Merged_multiome.rds')

In [ ]:
merged

In [ ]:
merged@assays

In [ ]:
unique(merged$Sample)

In [ ]:
unique(merged$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
merged$nucleosome_group <- ifelse(merged$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = merged, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="Merged_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(merged, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Merged_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(merged, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_vln_prefilter.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

# Doublets removal

In [ ]:
table(merged$orig.ident)

## RNA

In [ ]:
DefaultAssay(merged) <- "RNA"

Since we already have the peaks assay we can delete the ATAC assay, which allows to save computational resources 

In [ ]:
samples.list <- SplitObject(merged, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
pcas <- c()
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
i = 1
for(sample in samples.list){
    sample_n <- unique(sample$Sample)
    DefaultAssay(sample) <- 'RNA'
    sample <- NormalizeData(sample, normalization.method = "LogNormalize", scale.factor = 10000)
    sample <- FindVariableFeatures(sample, selection.method = "vst", nfeatures=4000) 
    sample <- ScaleData(sample) 
    sample <-  RunPCA(sample) 
    sample <-  FindNeighbors(sample, reduction = "pca", dims = 1:20)
    print(unique(sample$Sample))
    pca <- ElbowPlot(sample, ndims = 50)
    
    print(pca)

    plot_name = paste("", sample_n, "_pca.svg")

    
    ggsave(file=plot_name, plot=pca, width=6, height=4)
    
    samples.list[[i]] <- sample

    i <- i+1
}

In [ ]:
samples.list

In [ ]:
i = 1
for(sample in samples.list){
    sample_n <- unique(sample$Sample)
    DefaultAssay(sample) <- 'RNA'
    sample <- FindClusters(sample, resolution = 0.1)
    sample <- RunUMAP(sample, reduction = "pca", dims = 1:20)
    
    samples.list[[i]] <- sample

    i <- i+1
    
}

In [ ]:
print(paste("Sample : ", unique(samples.list[[1]]$Sample)))
sweep.res <- paramSweep_v3(samples.list[[1]], PCs = 1:20, sct = FALSE)
sweep.stats <- summarizeSweep(sweep.res, GT = FALSE)
bcmvn <- find.pK(sweep.stats)

print(bcmvn)

In [ ]:
print(paste("Sample : ", unique(samples.list[[2]]$Sample)))
sweep.res <- paramSweep_v3(samples.list[[2]], PCs = 1:20, sct = FALSE)
sweep.stats <- summarizeSweep(sweep.res, GT = FALSE)
bcmvn <- find.pK(sweep.stats)

print(bcmvn)

In [ ]:
print(paste("Sample : ", unique(samples.list[[3]]$Sample)))
sweep.res <- paramSweep_v3(samples.list[[3]], PCs = 1:20, sct = FALSE)
sweep.stats <- summarizeSweep(sweep.res, GT = FALSE)
bcmvn <- find.pK(sweep.stats)

print(bcmvn)

In [ ]:
print(paste("Sample : ", unique(samples.list[[4]]$Sample)))
sweep.res <- paramSweep_v3(samples.list[[4]], PCs = 1:20, sct = FALSE)
sweep.stats <- summarizeSweep(sweep.res, GT = FALSE)
bcmvn <- find.pK(sweep.stats)

print(bcmvn)

In [ ]:
print(paste("Sample : ", unique(samples.list[[5]]$Sample)))
sweep.res <- paramSweep_v3(samples.list[[5]], PCs = 1:20, sct = FALSE)
sweep.stats <- summarizeSweep(sweep.res, GT = FALSE)
bcmvn <- find.pK(sweep.stats)

print(bcmvn)

In [ ]:
print(paste("Sample : ", unique(samples.list[[6]]$Sample)))
sweep.res <- paramSweep_v3(samples.list[[6]], PCs = 1:20, sct = FALSE)
sweep.stats <- summarizeSweep(sweep.res, GT = FALSE)
bcmvn <- find.pK(sweep.stats)

print(bcmvn)

In [ ]:
samples.list

In [ ]:
pk_values <- c(0.04, 0.005, 0.28, 0.02, 0.25, 0.03)
dbl_rate <- c(0.12, 0.14, 0.09, 0.017, 0.029, 0.014)

In [ ]:
names(pk_values) <- names(samples.list)

In [ ]:
for (i in 1:length(samples.list)) {
    
    nEXP <- round(ncol(samples.list[[i]])*dbl_rate[[i]])

    pN_value <- 0.25

    samples.list[[i]] <- doubletFinder_v3(samples.list[[i]], PCs = 1:20, pN = pN_value, pK = pk_values[i], nExp = nEXP, reuse.pANN = FALSE, sct=FALSE)
}

In [ ]:
RNA_dbls_info <- c()
for (i in 1:length(samples.list)) {
    DF_tmp <- samples.list[[i]]@meta.data[,(ncol(samples.list[[i]]@meta.data)-1):ncol(samples.list[[i]]@meta.data)]
    colnames(DF_tmp) <- c("pANN", "DF.classifications")
    RNA_dbls_info <- rbind(RNA_dbls_info, DF_tmp)
}


In [ ]:
merged@meta.data[c("pANN", "DF.classifications")] <- RNA_dbls_info[colnames(merged[["RNA"]]), c("pANN", "DF.classifications")]

In [ ]:
saveRDS(merged, 'Merged_multiome_dbl.rds')

## ATAC

In [ ]:
merged <- readRDS('Merged_multiome_dbl.rds')

In [ ]:
samples.list <- SplitObject(merged, split.by = 'Sample')

In [ ]:
samples.list

In [ ]:
DefaultAssay(merged) <- 'ATAC'

***NOTE:*** For AMULET the *doublets_rate* parameter is the decimal equivalent of the percentage of expected doublets given by the 10X table. We used the number of yield cells to get the approximation of the expected percentage of doublets. We devided the percentage by 100 to get the decimal value to give as input in the function. https://user-images.githubusercontent.com/40582930/81216317-91f3f100-8f8f-11ea-9119-de8a158bc0a4.png

*EG.:* A data set with 2000 cells has a percentage of expected doublets of 1.6%. The value to the function parameter will be 1.6/100 = 0.016

***NOTE:*** To get the *RepeatMasker* file to use in AMULET we need to go to the following web page http://genome.ucsc.edu/cgi-bin/hgTables and select as follows:

In [ ]:
multiplets_ATAC <- c()

for (i in 1:length(samples.list)){
    sample_name <- unique(samples.list[[i]]$Sample)
    print(sample_name)
    
    path_to_multiplets_file <- paste0('', sample_name, '/AMULET_Results/MultipletCellIds_01.txt')
    
    multiplets_name <- paste(sample_name, 'multiplets')
    
    multiplets_ATAC <- c(multiplets_ATAC, assign(multiplets_name, read.table(path_to_multiplets_file)))
}

In [ ]:
ATAC_doublets_cells <- c()
for (i in 1:length(multiplets_ATAC)){
    ATAC_doublets_cells <- c(ATAC_doublets_cells, paste0(unique(samples.list[[i]]$Sample), "_", substr(multiplets_ATAC[[i]],1,nchar(multiplets_ATAC[[i]])-2), "-", 1))
}



In [ ]:
ATAC_doublets_cells[1:5]

In [ ]:
length(ATAC_doublets_cells)

In [ ]:
merged$Amulet_dbl_prediction <- rep("Singlet", nrow(merged@meta.data))

In [ ]:
merged$Amulet_dbl_prediction[1:5]

In [ ]:
for (i in 1:nrow(merged@meta.data)){
    if (rownames(merged@meta.data[i,]) %in% ATAC_doublets_cells){
        merged@meta.data[i,"Amulet_dbl_prediction"] <- "Doublet"
    }
}

In [ ]:
saveRDS(merged, 'Merged_multiome_dbl.rds')

In [ ]:
merged <- readRDS('Merged_multiome_dbl.rds')

In [ ]:
merged_dbl <- merged

In [ ]:
merged_dbl

In [ ]:
DefaultAssay(merged_dbl) <- 'RNA' 

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
merged_dbl <- merged_dbl %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(merged_dbl, ndims = 50)

pca

ggsave(file="merged_pca.svg", plot=pca, width=6, height=4)

In [ ]:
merged_dbl <- RunUMAP(merged_dbl, dims = 1:20, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(merged_dbl, reduction = 'umap', group.by = 'Sample', pt.size = 0.1, raster = TRUE, cols = my_c)

dpl

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(merged_dbl, reduction = 'umap', group.by = 'DF.classifications', pt.size = 0.1, raster = TRUE, cols = my_c)

dpl

ggsave(file="merged_dbl_RNA_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
DefaultAssay(merged_dbl) <- 'ATAC'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(merged_dbl, reduction = 'umap', group.by = 'Amulet_dbl_prediction', pt.size = 0.1, raster = TRUE, cols = my_c)

dpl

ggsave(file="merged_dbl_ATAC_UMAP.svg", plot=dpl, width=8, height=4)

# Remove doublets

In [ ]:
merged_dbl <-subset(merged_dbl,
                      subset=DF.classifications=='Singlet' &
                      Amulet_dbl_prediction=='Singlet')

In [ ]:
DefaultAssay(merged_dbl) <- 'RNA'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(merged_dbl, reduction = 'umap', group.by = 'DF.classifications', pt.size = 0.1, raster = TRUE, cols = my_c)

dpl

ggsave(file="merged_nodbl_RNA_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(merged_dbl, reduction = 'umap', group.by = 'Amulet_dbl_prediction', pt.size = 0.1, raster = TRUE, cols = my_c)

dpl

ggsave(file="merged_nodbl_ATAC_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(merged_dbl, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("", sample, "_vln_prefilter_nodbl.svg")

    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

**NOTE**:The functions *get_density* and *DensityScatter* are from the packages Signac V 1.10, since we are using the version 1.4 we copied only the necessary function to plot the wanted plots.

In [ ]:
get_density <- function(x, y, ...) {
  if (!requireNamespace("MASS", quietly = TRUE)) {
    stop("Please install MASS: install.packages('MASS')")
  }
  if (!is.numeric(x = x) | !is.numeric(x = y)) {
      stop("Must supply numeric values")
  }
  dens <- MASS::kde2d(x = x, y = y, ...)
  ix <- findInterval(x = x, vec = dens$x)
  iy <- findInterval(x = y, vec = dens$y)
  ii <- cbind(ix, iy)
  return(dens$z[ii])
}

In [ ]:
DensityScatter <- function(
    object,
    x,
    y,
    log_x = FALSE,
    log_y = FALSE,
    quantiles = NULL
) {
    md <- object[[]]
    if (!(x %in% colnames(x = md))) {
        stop(x, " not found")
    }
    if (!(y %in% colnames(x = md))) {
        stop(y, " not found")
    }
    log10p <- function(x) {
        return(log10(x = x + 1))
    }
    null_fn <- function(x) {
        return(x)
    }
    logfnx <- ifelse(test = log_x, yes = log10p, no = null_fn)
    logfny <- ifelse(test = log_y, yes = log10p, no = null_fn)

    md$Density <- get_density(
        x = logfnx(md[[x]]),
        y = logfny(md[[y]]),
        n = 1000,
        h = c(1,1)
    )
    md <- md[order(md$Density), ]
    # quantiles
    use_quantile <- FALSE
    if (!is.null(x = quantiles)) {
        use_quantile <- TRUE
        # set default if TRUE passed
        if (is.logical(x = quantiles)) {
            if (quantiles) {
                quantiles <- c(5, 10, 90, 95)
            } else {
                use_quantile <- FALSE
            }
        }
        # make sure integers between 0 and 100
        if (!(all(quantiles >= 0) & all(quantiles <= 100))) {
            warning("Quantile values must be between 0 and 100",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
        if (any(sapply(X = quantiles, FUN = function(x) x %% 1 != 0))) {
            warning("Quantile values must be integers",
                    immediate. = TRUE)
            use_quantile <- FALSE
        }
    }
    if (use_quantile) {
        # convert to string
        quantiles <- paste0(quantiles, '%')

        # quantiles for x and y
        x_quant <- quantile(x = md[[x]], probs = seq(0, 1, 0.01))
        y_quant <- quantile(x = md[[y]], probs = seq(0, 1, 0.01))
        xlines <- x_quant[quantiles]
        ylines <- y_quant[quantiles]

        # round
        xlines <- round(x = xlines, digits = 2)
        ylines <- round(x = ylines, digits = 2)
    }        
    p <- ggplot(
           data = md,
           mapping = aes(x = .data[[x]], y = .data[[y]], color = .data[["Density"]])
         ) +
           geom_point(size = 1) +
           scale_color_viridis_c(option = "B") +
           theme_bw()
    if (log_x) {
        p <- p + scale_x_log10()
    }
    if (log_y) {
        p <- p + scale_y_log10()
    }
    if (use_quantile) {
        p <- p +
          geom_vline(xintercept = unname(obj = xlines), color = "red") +
          geom_hline(yintercept = unname(obj = ylines), color = "red") +
          labs(title = "Quantiles",
               subtitle = paste0(x, ": ", paste0(names(x = xlines), ":",
                                                 xlines, collapse = " "), "\n",
                                 y, ": ", paste0(names(x = ylines), ":",
                                                 ylines, collapse = " ")))
    }
    return(p)
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(merged_dbl, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_RNA', y = 'nFeature_RNA', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl_RNA.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(merged_dbl, subset = Sample == sample)
    scatterplt <- DensityScatter(object = s, x = 'nCount_ATAC', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)
    
    plot_name = paste("", sample, "Densenty_scatter_prefilter_nodbl.svg")

    options(repr.plot.width = 8, repr.plot.height = 6)
    print(scatterplt)
    
    ggsave(file=plot_name, plot=scatterplt, width=8, height=6)
    
    
}

In [ ]:
saveRDS(merged_dbl, 'Merged_noDBL_multiome.rds')

# Filtering

In [ ]:
merged_dbl <- readRDS('Merged_noDBL_multiome.rds')

In [ ]:
nCount_RNA_max <- c(12500, 12500, 13000, 28000, 60000, 5000)

nCount_RNA_min <- c(500, 650, 600, 500, 700, 200) 

nFeature_RNA_max <- c(5500, 5000, 5500, 6200, 9000, 2500) 

nFeature_RNA_min <- c(250, 250, 250, 250, 250, 250)

nCount_ATAC_max <- c(20000, 20000, 20000, 25000, 70000, 9000)

nCount_ATAC_min <- c(400, 400, 400, 450, 600, 200) 

nFeature_ATAC_max <- c(15000, 16000, 18000, 20000, 50000, 6000)

nFeature_ATAC_min <- c(250, 250, 250, 250, 250, 250)

nucleosome_signal_max <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0) 

TSS.enrichment_min <- c(2.0, 2.0, 2.0, 2.0, 2.0, 2.0) 

pct_mito_max <- c(0.3, 0.3, 0.3, 1.3, 0.6, 1.2) 

pct_hb_max <- c(0.075, 0.075, 0.06, 0.05, 0.05, 0.05)

In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
cells_to_keep <- c()

In [ ]:
for (i in 1:length(sample_names)){
    sample <- sample_names[i]
    print(sample)

    s <- subset(merged_dbl, subset = Sample == sample)

    print(s)
    
    print(nCount_RNA_max[i])
    print(nCount_ATAC_max[i])
    print(nCount_RNA_min[i])
    print(nCount_ATAC_min[i])
    print(nFeature_RNA_max[i])
    print(nFeature_ATAC_max[i])
    print(nFeature_RNA_min[i])
    print(nFeature_ATAC_min[i])
    print(nucleosome_signal_max[i])
    print(TSS.enrichment_min[i])
    print(pct_mito_max[i])
    print(pct_hb_max[i])
      
      
    s <-  subset(s, subset=nCount_RNA < nCount_RNA_max[i] &
                      nCount_ATAC < as.numeric(nCount_ATAC_max[i]) &
                      nCount_RNA > as.numeric(nCount_RNA_min[i]) &
                      nCount_ATAC > as.numeric(nCount_ATAC_min[i]) &
                      nFeature_RNA < as.numeric(nFeature_RNA_max[i]) &
                      nFeature_ATAC < as.numeric(nFeature_ATAC_max[i]) &
                      nFeature_RNA > as.numeric(nFeature_RNA_min[i]) &
                      nFeature_ATAC > as.numeric(nFeature_ATAC_min[i]) &
                      nucleosome_signal < as.numeric(nucleosome_signal_max[i]) &
                      TSS.enrichment > as.numeric(TSS.enrichment_min[i]) &
                      pct_mito < as.numeric(pct_mito_max[i]) &
                      pct_hb < as.numeric(pct_hb_max[i]))
    
    cells_to_keep <- c(cells_to_keep, colnames(s))
    
    print(length(colnames(s)))
    print(length(cells_to_keep))
    
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    print(vln)

    plot_name = paste("", sample, "_vln_afterfilter.svg")

    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
}

In [ ]:
merged_filt <- subset(merged_dbl, cells = cells_to_keep)

In [ ]:
table(merged_filt$orig.ident)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(merged_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="Merged_all_vln_after_filter.svg", plot=vln, width=15, height=8)


In [ ]:
saveRDS(merged_filt, 'Merged_filt_multiome.rds')

# Call peaks

In [ ]:
merged_filt <- readRDS('Merged_filt_multiome.rds')

In [ ]:
merged_filt

In [ ]:
DefaultAssay(merged_filt) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=merged_filt,
                 macs2.path = 'macs2', 
                outdir='callpeaks', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(merged_filt), features = peaks, cells = colnames(merged_filt))

In [ ]:
merged_filt[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(merged_filt), annotation = annotation)

In [ ]:
merged_filt@assays

In [ ]:
saveRDS(merged_filt, 'Merged_filt_multiome_clusters_peaks.rds')

# FRiP

In [ ]:
merged_filt <- FRiP(merged_filt, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(merged_filt, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="Merged_filt_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(merged_filt, 'Merged_filt_multiome_clusters_peaks.rds')

# Integration and clustering

In [ ]:
merged_filt <- readRDS('Merged_filt_multiome_clusters_peaks.rds')

## RNA

In [ ]:
table(merged_filt$orig.ident)

In [ ]:
DefaultAssay(merged_filt) <- 'RNA'

In [ ]:
merged_filt <- merged_filt %>% 
                            NormalizeData() %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(merged_filt, ndims = 50)

pca

ggsave(file="Merged_pca.svg", plot=pca, width=6, height=4)

In [ ]:
merged_filt <- RunUMAP(merged_filt, dims = 1:20, reduction = 'pca', n.neighbors = 15, reduction.name="umapRNA")

In [ ]:
unique(merged_filt$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(merged_filt, reduction = 'umapRNA', group.by = 'Sample', pt.size = 0.1, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="Merged_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(merged_filt, reduction = 'umapRNA', group.by = 'Age', pt.size = 0.1, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Merged_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(merged_filt, reduction = 'umapRNA', group.by = 'Batch', pt.size = 0.1, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="Merged_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(merged_filt, group.by="orig.ident", split.by="orig.ident", reduction="umapRNA", shuffle=TRUE, cols=my_c, ncol=3, pt.size=0.1, label=FALSE)

dpl


ggsave(file="Merged_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 35, repr.plot.res = 200)
ftp <- FeaturePlot(merged_filt, reduction = 'umapRNA', features = c('eGFP', 'Ttr', 'Clic6', 'Folr1', 'Gata3', 'Pon1', 'Gpx3', 'Atp10b', 'Cp', 'Cidea', 'Tbc1d1', 'Rian', 'Shisa8', 'Deup1', 'Pecam1', 'Acta2', 'Plvap', 
                                                  'Dcn', 'Lum', 'Plp1', 'Sox10', 'Mbp', 'Mog', 'C1qa', 'C1qb', 'Ptprc', 'Top2a', 'Mki67', 'Mpz'), 
             min.cutoff = 0, max.cutoff = 4, label.size = 2, ncol = 4, pt.size = 0.2, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Merged_geneExpression.svg", plot=ftp, width=20, height=35)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(merged_filt, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated

In [ ]:
integrated <- RunUMAP(integrated, reduction = 'harmonyRNA', dims = 1:20, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 0.1, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="Harmony_RNA_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 0.2, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="Harmony_RNA_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 0.2, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="Harmony_RNA_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)


In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, 
               cols=my_c, ncol=3, pt.size=0.2, label=FALSE)

dpl

ggsave(file="Harmony_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

### Clustering

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- integrated %>% FindNeighbors(reduction = 'harmonyRNA', dims = 1:20) %>%
                                FindClusters(resolution = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 300)
grid <- plot_grid(DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.1', cols = my_c, raster = TRUE), 
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.2', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.3', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.4', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.5', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.6', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.7', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.8', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.0.9', cols = my_c, raster = TRUE),
                  DimPlot(integrated, reduction = "harmony_RNA", pt.size=0.1, group.by = 'RNA_snn_res.1', cols = my_c, raster = TRUE))
grid

ggsave(file="Integrated_clusters_all.svg", plot=grid, width=15, height=30)

In [ ]:
saveRDS(integrated, 'Merged_filt_multiome_clusters_peaks.rds')

# Peaks

In [ ]:
integrated <- readRDS('Merged_filt_multiome_clusters_peaks.rds')

## Normalization and dimensional reduction

In [ ]:
DefaultAssay(integrated)<-'peaks'
integrated<-FindTopFeatures(integrated,min.cutoff = NULL)
integrated<-RunTFIDF(integrated) #Run term frequency inverse document frequency (TF-IDF) normalization on a matrix.
integrated<-RunSVD(integrated)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5, repr.plot.res = 200)
DepthCor(integrated,n=50,reduction = 'lsi')

In [ ]:
options(repr.plot.width=6, repr.plot.height=4)
ebl <- ElbowPlot(integrated, reduction = "lsi", n=50)

ebl

ggsave(file="Merged_depthCorr_ebl.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated <- RunUMAP(integrated,reduction = 'lsi',dims = 2:30,reduction.name = 'atac.umap')
integrated <- FindNeighbors(integrated,reduction = 'lsi',dims = 2:30)

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Sample", reduction="atac.umap", pt.size = 0.1, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="integrated_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

## Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(integrated, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'peaks', reduction.save = 'harmonyPeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated <- RunUMAP(integrated, dims = 2:30, reduction = 'harmonyPeaks', reduction.name = 'harmony_Peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by = 'Sample', pt.size = 0.1, raster=TRUE, cols=my_c, reduction = 'harmony_Peaks')

dpl

ggsave(file="integrated_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=0.1, raster = TRUE, label=FALSE)

dpl

ggsave(file="integrated_Peaks_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Batch", reduction="harmony_Peaks", pt.size = 0.1, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="integrated_Peaks_Batch_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="Batch", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=5, pt.size=0.1, raster = TRUE, label=FALSE)

dpl

ggsave(file="integrated_Peaks_each_batch_UMAP.svg", plot=dpl, width=16, height=6)

In [ ]:
saveRDS(integrated, 'Integrated_multiome_clusters_peaks.rds')

# Joint vizualization

In [ ]:
integrated <- readRDS('Integrated_multiome_clusters_peaks.rds')

In [ ]:
integrated@reductions

In [ ]:
integrated

In [ ]:
integrated <- FindMultiModalNeighbors(integrated, reduction.list = c('harmonyRNA', 'harmonyPeaks'),
                                       dims.list = list(1:20,2:30), verbose=TRUE)

In [ ]:
integrated <- RunUMAP(integrated, nn.name = "weighted.nn", assay = "RNA", reduction.name = 'multiome.umap',  verbose = TRUE)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Sample", pt.size=0.1, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'multiome.umap')

dpt

ggsave(file="integrated_clusters_Sample_Multiome.svg", plot=dpt, width=7, height=5)

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 35, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'multiome.umap', features = c('eGFP', 'Ttr', 'Clic6', 'Folr1', 'Gata3', 'Pon1', 'Gpx3', 'Atp10b', 'Cp', 'Cidea', 'Tbc1d1', 'Rian', 'Shisa8', 'Deup1', 'Pecam1', 'Acta2', 'Plvap', 
                                                  'Dcn', 'Lum', 'Gria2', 'Sox10', 'Mbp', 'Ptprz1', 'C1qa', 'Cd86', 'Ptprc', 'Top2a', 'Mki67', 'Mpz'), 
             min.cutoff = 0, label.size = 2, ncol = 4, pt.size = 0.2, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="Joint_geneExpression.svg", plot=ftp, width=20, height=35)

In [ ]:
saveRDS(integrated, 'Integrated_multiome_clusters_joint.rds')

# Transfer label

In [ ]:
integrated <- readRDS('Integrated_multiome_clusters_joint.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

The following file is the result of the scRNA-seq analysis.

In [ ]:
sc_rna_annotated <- readRDS('noDBL_Postnatal_noDBL_filtered_clustered_annotated.rds')

In [ ]:
sc_rna_annotated

In [ ]:
sc_rna_annotated <- sc_rna_annotated %>%
    NormalizeData() %>%
    FindVariableFeatures() %>%
    ScaleData() %>%
    RunPCA(verbose = F) %>%
    RunUMAP(dims = 1:20,verbose=FALSE)

In [ ]:
sc_rna_annotated

In [ ]:
table(sc_rna_annotated$Cell_type)

In [ ]:
transfer.anchors <- FindTransferAnchors(reference = sc_rna_annotated, query = integrated, dims = 1:20)
predictions <- TransferData(anchorset = transfer.anchors, refdata = sc_rna_annotated$Cell_type, dims = 1:20)
integrated <- AddMetaData(object = integrated, metadata = predictions)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "predicted.id", pt.size=0.1, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="Integrated_clusters_predictedid_RNA.svg", plot=dpt, width=7, height=5)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "predicted.id", pt.size=0.1, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_Peaks')

dpt

ggsave(file="Integrated_clusters_predictedid_Peaks.svg", plot=dpt, width=7, height=5)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "predicted.id", pt.size=0.1, raster = TRUE, cols = my_c, label = FALSE, 
               repel = TRUE, reduction = 'multiome.umap')

dpt

ggsave(file="Integrated_clusters_predictedid_joint.svg", plot=dpt, width=7, height=5)

In [ ]:
integrated <- SetIdent(integrated, value = 'predicted.id')

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
write.csv(integrated_predicted.markers, 'predicted_id_Markers_RNA_all.csv', row.names = TRUE)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

ggsave(file="Integrated_dotplot_predictedid.svg", plot=dtp, width=10, height=6)

In [ ]:
marker_genes <- c('Folr1', 'Clic6', 'Krt18', 'Dcn', 'Itga8', 'Col3a1', 'Flt1', 'Plvap', 'Pecam1', 'Cd86', 'Tank', 
                  'Cd83', 'Dscam', 'Ptprz1', 'Gria2')

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 20, repr.plot.res = 300)
vln <- VlnPlot(object = integrated, features = marker_genes, pt.size=0, cols = my_c, ncol = 4)

vln


In [ ]:
modify_vlnplot<- function(obj, feature, pt.size = 0, 
                          plot.margin = unit(c(-0.75, 0, -0.75, 0), "cm"),
                          ...) {
  p<- VlnPlot(obj, features = feature, pt.size = pt.size, ... )  + 
    xlab("") + ylab(feature) + ggtitle("") + 
    theme(legend.position = "none", 
          axis.text.x = element_blank(), 
          axis.ticks.x = element_blank(),
          axis.title.y = element_text(size = rel(1), angle = 0), 
          axis.text.y = element_text(size = rel(1)), 
          plot.margin = plot.margin ) 
  return(p)
}

In [ ]:
extract_max<- function(p){
  ymax<- max(ggplot_build(p)$layout$panel_scales_y[[1]]$range$range)
  return(ceiling(ymax))
}

In [ ]:
StackedVlnPlot<- function(obj, features,
                          pt.size = 0, 
                          plot.margin = unit(c(-0.75, 0, -0.75, 0), "cm"),
                          ...) {
  
  plot_list<- purrr::map(features, function(x) modify_vlnplot(obj = obj,feature = x, ...))
  
  # Add back x-axis title to bottom plot. patchwork is going to support this?
  plot_list[[length(plot_list)]]<- plot_list[[length(plot_list)]] +
    theme(axis.text.x=element_text(angle = 45, hjust = 1, vjust = 1), 
          axis.ticks.x = element_line())
  
  # change the y-axis tick to only max value 
  ymaxs<- purrr::map_dbl(plot_list, extract_max)
  plot_list<- purrr::map2(plot_list, ymaxs, function(x,y) x + 
                            scale_y_continuous(breaks = c(y)) + 
                            expand_limits(y = y))
  
  p<- patchwork::wrap_plots(plotlist = plot_list, ncol = 1)
  return(p)
}

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 30, repr.plot.res = 100)

svln <- StackedVlnPlot(integrated,  features = marker_genes, cols = my_c)

svln 

ggsave(file="Integrated_svln_Cell_type_predicted_id.svg", plot=svln, width=6, height=30)

In [ ]:
saveRDS(integrated, 'Integrated_multiome_clusters_genes.rds')

# Link peaks to genes

In [ ]:
integrated <- readRDS('Integrated_multiome_clusters_genes.rds')

In [ ]:
library(BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
# first compute the GC content for each peak
integrated <- RegionStats(integrated, genome = BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
integrated[['ATAC']] <- NULL

In [ ]:
integrated

In [ ]:
integrated <- LinkPeaks(integrated, peak.assay = "peaks", expression.assay = "RNA")

In [ ]:
integrated

In [ ]:
saveRDS(integrated, 'Integrated_multiome_clusters_joint_linked.rds')

# Cell type annotation

In [ ]:
integrated <- readRDS('Integrated_multiome_clusters_joint_linked.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated <- SetIdent(integrated, value = 'RNA_snn_res.0.6')

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

In [ ]:
integrated@meta.data$Cell_type <- 'placeholder'

In [ ]:
integrated@meta.data$RNA_snn_res.0.6 <- as.factor(integrated@meta.data$RNA_snn_res.0.6)

In [ ]:
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "0")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "1")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "2")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "3")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "4")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "5")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "6")] <- "Mesenchymal cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "7")] <- "Endothelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "8")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "9")] <- "Epithelial cells"
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "10")] <- "Mesenchymal cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "11")] <- "Epithelial cells"
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "12")] <- "Immune cells"
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "13")] <- "Epithelial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "14")] <- "Neuronal and glial cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "15")] <- "Mesenchymal cells" 
integrated@meta.data$Cell_type[which(integrated@meta.data$RNA_snn_res.0.6 == "16")] <- "Endothelial cells" 

In [ ]:
integrated <- SetIdent(integrated, value = 'Cell_type')

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
write.csv(integrated_predicted.markers, 'Markers_RNA_All.csv', row.names = TRUE)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted_noDBL

In [ ]:
top10_predicted_noDBL <- top10_predicted_noDBL[!duplicated(top10_predicted_noDBL$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

ggsave(file="All_integrated_dotplot_Cell_type.svg", plot=dtp, width=10, height=6)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Cell_type", pt.size=0.1, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_RNA')

dpt

ggsave(file="All_integrated_Cell_type_RNA.svg", plot=dpt, width=7, height=4)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Cell_type", pt.size=0.1, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'harmony_Peaks')

dpt

ggsave(file="All_integrated_Cell_type_Peaks.svg", plot=dpt, width=7, height=4)

In [ ]:
integrated

In [ ]:
saveRDS(integrated, 'Integrated_multiome_clusters_joint_linked_cell_type.rds')